# Mouna — multi-seed sweep: v2norm (no motion/orientation) vs v3motion (with motion/orientation)

Runs all 10 training runs (5 seeds x 2 configs) on GPU and reports the real, paired-statistical-test verdict at the end — same discipline used for every comparison in this project. Takes roughly 10-20 min total on a Colab GPU (each run is a few minutes; our local MPS runs took longer).

**Before running:** `Runtime -> Change runtime type -> GPU`.

**One manual step first:** upload BOTH dataset files to Google Drive, in a folder called `mouna_data/`:
- `keypoints_official_split_v2norm.parquet` (204MB) — baseline, per-part normalization only
- `keypoints_official_split_v3motion.parquet` (401MB) — same data + orientation/velocity features

Both are at `~/mouna/mouna/data/silver/keypoints/` on your Mac.

In [ ]:
# 1. Clone the repo
!git clone https://github.com/vimalthomas/mouna.git
%cd mouna

In [ ]:
# 2. Install deps
!pip install -q torch pandas pyarrow scikit-learn mlflow scipy

In [ ]:
# 3. Mount Drive and copy both datasets into place
from google.colab import drive
drive.mount('/content/drive')

import os
os.makedirs('data/silver/keypoints', exist_ok=True)
os.makedirs('data/sweep', exist_ok=True)
!cp /content/drive/MyDrive/mouna_data/keypoints_official_split_v2norm.parquet data/silver/keypoints/
!cp /content/drive/MyDrive/mouna_data/keypoints_official_split_v3motion.parquet data/silver/keypoints/
!ls -lh data/silver/keypoints/

In [ ]:
# 4. Confirm GPU is actually visible
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('No GPU detected — go to Runtime > Change runtime type > GPU, then rerun from cell 1.')

In [ ]:
# 5. Run the full sweep: 5 seeds x 2 configs = 10 runs
import os, re, subprocess, time

SEEDS = [42, 1, 2, 3, 4]
CONFIGS = {
    'v2norm':   'data/silver/keypoints/keypoints_official_split_v2norm.parquet',
    'v3motion': 'data/silver/keypoints/keypoints_official_split_v3motion.parquet',
}

results = {}  # {(config, seed): test_acc}

for config_name, parquet_path in CONFIGS.items():
    for seed in SEEDS:
        env = os.environ.copy()
        env.update({
            'PYTHONPATH': 'src',
            'MLFLOW_ALLOW_FILE_STORE': 'true',
            'MOUNA_PARQUET': parquet_path,
            'MOUNA_MODEL_OUT': f'data/sweep/{config_name}_seed{seed}.pth',
            'MOUNA_RUN_NAME': f'colab-sweep-{config_name}-seed{seed}',
            'MOUNA_DATASET_TAG': f'colab-sweep-{config_name}',
            'MOUNA_SEED': str(seed),
        })
        print(f'=== {config_name}, seed {seed} ===', flush=True)
        t0 = time.time()
        proc = subprocess.run(
            ['python3', '-u', 'scripts/train_transformer.py'],
            env=env, capture_output=True, text=True,
        )
        log = proc.stdout + proc.stderr
        m = re.search(r'Test acc: ([\d.]+)%', log)
        acc = float(m.group(1)) if m else None
        results[(config_name, seed)] = acc
        elapsed = time.time() - t0
        print(f'  -> test_acc={acc}%  ({elapsed:.0f}s)', flush=True)
        if acc is None:
            print('  !! no test_acc parsed — last 2000 chars of log:')
            print(log[-2000:])

print()
print('All results:', results)

In [ ]:
# 6. The real verdict — paired t-test + Wilcoxon, same as every other comparison in this project
import numpy as np
from scipy import stats

v2norm = np.array([results[('v2norm', s)] for s in SEEDS])
v3motion = np.array([results[('v3motion', s)] for s in SEEDS])
deltas = v3motion - v2norm

print('seed   v2norm   v3motion   delta')
for s, a, b in zip(SEEDS, v2norm, v3motion):
    print(f'{s:4d}  {a:6.1f}  {b:8.1f}   {b-a:+.1f}')
print()
print(f'v2norm    mean={v2norm.mean():.2f}  std={v2norm.std(ddof=1):.2f}')
print(f'v3motion  mean={v3motion.mean():.2f}  std={v3motion.std(ddof=1):.2f}')
print(f'delta (v3motion-v2norm) mean={deltas.mean():.2f}  std={deltas.std(ddof=1):.2f}')
print(f'v3motion wins: {(deltas>0).sum()}/{len(SEEDS)} seeds')

t, p_t = stats.ttest_rel(v3motion, v2norm)
w, p_w = stats.wilcoxon(deltas)
print(f'\\npaired t-test: t={t:.3f}  p={p_t:.4f}')
print(f'wilcoxon signed-rank: p={p_w:.4f}')

sig = (p_t < 0.05) or (p_w < 0.05)
direction = 'beats' if deltas.mean() > 0 else 'LOSES TO'
print()
if sig:
    print(f'=> SIGNIFICANT at p<0.05 — v3motion (motion/orientation features) {direction} v2norm '
          f'(mean delta {deltas.mean():+.2f}pp). Real effect, not noise.')
else:
    print(f'=> NOT significant (p_t={p_t:.3f}, p_w={p_w:.3f}) — indistinguishable from noise. '
          f'Mean delta {deltas.mean():+.2f}pp, direction: v3motion {direction} v2norm, but NOT confirmed.')

In [ ]:
# 7. Download all 10 checkpoints + a results summary back to your Mac
import json
with open('data/sweep/results.json', 'w') as f:
    json.dump({f'{k[0]}_seed{k[1]}': v for k, v in results.items()}, f, indent=2)

!zip -r sweep_results.zip data/sweep/
from google.colab import files
files.download('sweep_results.zip')